# 04 — Match McMaster

| | |
|---|---|
| **Pipeline stage** | `match_mcmaster` |
| **Service module** | `backend.services.pipeline.match_parts_only` → `matcher.match_parts` |
| **Website equivalent** | `match_mcmaster` events in `POST /api/import/stream` |
| **Prev / Next** | `03_parse_bom.ipynb` → `05_api_payload.ipynb` |

Calls `pipeline.match_parts_only` only — same as the website **before** `enrich_mcmaster`. Live SKU/pricing hydration runs in `05_api_payload.ipynb` / `import_from_url`.

**Structured hardware matching** (see `guess_strategy.py` + `metacategories.py`):

1. **McMaster department** — BOM intent maps to a metacategory (`Fastening & Joining`, `Power Transmission`, …) via `data/mcmaster_metacategories.json`
2. **Size filter** — metric (`thread-size~m3/length~16-mm`) or imperial (`system-of-measurement~inch/thread-size~#6-32`) narrows tables inside that department
3. **Primary guess** — high-confidence filtered browse to the simplest default material
4. **Same-size secondaries** — other finishes/materials or catalog SKUs at that thread/length
5. **Wider-scope secondaries** — cross-department or less-filtered browse paths


In [ ]:
from backend.models.part import Part

from backend.notebook_utils import (
    alternatives_to_dataframe,
    load_cached_parts,
    notebook_progress,
    parts_to_dataframe,
    prepare_crawl_env,
    resolve_parts_offline,
)
from backend.services.pipeline import match_parts_only
from backend.services.parsers.helpers.specification_checks import (
    check_parts_specifications,
    format_specification_issues,
)

prepare_crawl_env(reload_backend=False)
progress = notebook_progress("04")

parts = load_cached_parts()
if not parts:
    parts, _ = await resolve_parts_offline()

if not parts:
    from backend.services.vendors.mcmaster.cross_test import load_query_accuracy_cases

    print("Offline demo — parts from tests/fixtures/bom_listing_query_cases.json")
    parts = [case.part for case in load_query_accuracy_cases()[:8]]

spec_issues = check_parts_specifications(parts)
if spec_issues:
    print(format_specification_issues(spec_issues))

matched = match_parts_only(parts, on_progress=progress)
display(parts_to_dataframe(matched))
alts = alternatives_to_dataframe(matched)
if not alts.empty:
    print("\nSecondary guesses (same_size → wider_scope):")
    display(alts)


## Hardware family spot-check

Curated lines exercising nut subtypes, washers, and imperial sizing — same matcher as the website.

In [ ]:
from backend.models.part import Part
from backend.notebook_utils import alternatives_to_dataframe, parts_to_dataframe
from backend.services.pipeline import match_parts_only

spot_check = [
    Part(original_name="M3x16 socket head cap screw"),
    Part(original_name="M3 Nut"),
    Part(original_name="M3 nylock"),
    Part(original_name="M4 flat washer"),
    Part(original_name='#6-32 x 1/2"'),
    Part(original_name="M3 Hex Nut", specification="18-8 stainless"),
]

demo = match_parts_only(spot_check)
display(parts_to_dataframe(demo))
display(alternatives_to_dataframe(demo))